In [2]:
quiet_library <- function(...){suppressPackageStartupMessages(library(...))}
quiet_library(dplyr)
quiet_library(purrr)
quiet_library(tidyr)
quiet_library(data.table)
quiet_library(Seurat)
quiet_library(SeuratDisk)
quiet_library(ggplot2)
quiet_library(glue)
quiet_library(hise)
quiet_library(gridExtra)
quiet_library(grid)
quiet_library(fgsea)
quiet_library(ComplexHeatmap)
quiet_library(anndata)
quiet_library(RcppML)

options(repr.matrix.max.cols=150, repr.matrix.max.rows=200, mc.cores = 1, future.globals.maxSize = 2000 * 1024^2)
fig.size <- function (height, width) {
    options(repr.plot.height = height, repr.plot.width = width)
}

Warning message:
“package ‘RcppML’ was built under R version 4.3.3”


In [3]:
wd <- "/home/workspace/IFN/"
setwd(wd)

In [4]:
stims <- c("IFNa", "IFNg")
l1_celltypes <- c("Monocyte", "NK", "Tcell", "Bcell")
l2_celltypes <- c("Monocyte", "B_Naive", "B_Memory", "Plasma", 
             "CD4_Naive", "CD4_Memory", "CD8_Naive", "CD8_Memory", "gdT", "Treg", "MAIT", "NK.CD56dim", "NK.CD56hi")

In [5]:
cohorts <- c("ALPS", "MM", "SLE_Science", "SLE", "Acute_Covid", "Long_Covid", "Flu", "Malaria", 
            "RA", "MM_BMMC", "RA_external", "BRI_aging_M", "BRI_aging_F", "MM_Pretreatment", "MM_EndInduction")

In [6]:
meta <- fread(file.path(wd, "Cohort_IFN_scores/IFN_Cohort_Comparison_Sample_Sheet.csv"))

In [7]:
# all ISGs for scoring
#isgs <- fread(file.path(wd, "DEGs/IFNa_IFNg_all_bulk_subtype_DEGs.csv"))$ifn_genes
isgs <- fread(file.path(wd, "DEGs/IFNb_IFNg_all_bulk_subtype_DEGs.csv"))$ifn_genes

### Calculate Donor Differentials for Scoring ISGs

#### L1/L2

In [8]:
celltype_level <- "L1"
if (celltype_level == "L1"){celltypes <- l1_celltypes}
if (celltype_level == "L2"){celltypes <- l2_celltypes}

In [10]:
so <- readRDS(file.path(wd, "Cohort_IFN_scores", "Seurat_objects", glue("{cohort}_so.rds")))

In [ ]:
for (cohort in cohorts){
    # read in cohort object
    so <- readRDS(file.path(wd, "Cohort_IFN_scores", "Seurat_objects", glue("{cohort}_so.rds")))
    isgs_use <- intersect(isgs, rownames(so@assays$RNA))

    # append metadata
    meta_select <- meta %>% filter(Cohort == cohort) %>% select(KitID, Group, Cohort, Misc)
    common_cols <- intersect(colnames(so@meta.data), c("Group", "Cohort", "Misc"))
    so@meta.data <- so@meta.data %>% select(-all_of(common_cols)) %>% left_join(meta_select, 
                                                               by = "KitID")
    rownames(so@meta.data) <- colnames(so@assays$RNA@data)

    # define cohort donors and healthy sets
    donors <- meta_select %>% filter(Cohort == cohort & Group == cohort) %>% pull(KitID)
    healthies <- meta_select %>% filter(Cohort == cohort & Group == "Control") %>% pull(KitID)
    
    # iterate across cell types 
    for (c in celltypes){
        if (celltype_level == "L1"){sub_cell <- so %>% subset(celltype.l1 == c)}
        if (celltype_level == "L1"){sub_cell <- so %>% subset(celltype.l2 == c)}

        # iterate across donors
        for (d in donors){
            sub <- sub_cell %>% subset(KitID %in% c(d, healthies))
            sub$Group <- ifelse(sub$KitID == d, cohort, "Control")

            # ensure donor in object and sufficient cell numbers
            if (d %in% sub$KitID & table(sub$KitID)[d] > 30){
                degs <- FindMarkers(sub, ident.1 = cohort, ident.2 = "Control", logfc.threshold = 0, min.pct = 0.01, features = isgs_use,
                      group.by = "Group", assay = "RNA", test.use = "MAST") 
                degs$gene <- rownames(degs)
            
                degs %>% fwrite(file.path(wd, "Cohort_IFN_scores", "Cohort_DEGs", cohort, celltype_level, glue("{c}_{d}_MAST_degs.csv")))
                } 
            
            }
        }
    }

##### MM BMMC

In [ ]:
### generate "background" IFN scores for internal MM BMMC samples 

In [97]:
donors <- meta %>% filter(Group == "Control" & Cohort == "MM_BMMC_Internal") %>% pull(KitID)

In [ ]:
for (cohort in "MM_BMMC_Internal_Healthy"){

    so <- readRDS(file.path(wd, "Cohort_IFN_scores", "Seurat_objects", glue("{cohort}_so.rds"))) %>%
        subset(KitID %in% donors)
    isgs_use <- intersect(isgs, rownames(so@assays$RNA))
    meta_select <- meta %>% filter(Cohort == cohort) %>% select(KitID, Group, Cohort, Misc)
    common_cols <- intersect(colnames(so@meta.data), c("Group", "Cohort", "Misc"))
    so@meta.data <- so@meta.data %>% select(-all_of(common_cols)) %>% left_join(meta_select, 
                                                               by = "KitID")
    
    rownames(so@meta.data) <- colnames(so@assays$RNA@data)
    
    cohort <- "MM_BMMC_Internal_Healthy"
    for (c in bulk_celltypes){
        sub_cell <- so %>% subset(celltype.l1 == c)
        
        for (d in donors){
            donor_sub <- sub_cell %>% subset(KitID == d)
            donor_sub$Group <- cohort
            sub_cell$Group <- "Control"
            sub <- merge(donor_sub, sub_cell)
            
            sub$Group <- ifelse(sub$KitID == d, cohort, "Control")
            
            if (d %in% sub$KitID & table(sub$KitID)[d] > 30){
                degs <- FindMarkers(sub, ident.1 = cohort, ident.2 = "Control", logfc.threshold = 0, min.pct = 0.01, features = isgs_use,
                      group.by = "Group", assay = "RNA", test.use = "MAST") 
                degs$gene <- rownames(degs)
            
                degs %>% fwrite(file.path(wd, "Cohort_IFN_scores", "Cohort_DEGs", cohort, glue("{c}_{d}_MAST_degs.csv")))
                } 
            }
        }
    }

#### PBMC

In [ ]:
for (cohort in cohorts){
    # read in cohort object
    so <- readRDS(file.path(wd, "Cohort_IFN_scores", "Seurat_objects", glue("{cohort}_so.rds")))
    isgs_use <- intersect(isgs, rownames(so@assays$RNA))

    # append metadata
    meta_select <- meta %>% filter(Cohort == cohort) %>% select(KitID, Group, Cohort, Misc)
    common_cols <- intersect(colnames(so@meta.data), c("Group", "Cohort", "Misc"))
    so@meta.data <- so@meta.data %>% select(-all_of(common_cols)) %>% left_join(meta_select, 
                                                               by = "KitID")
    rownames(so@meta.data) <- colnames(so@assays$RNA@data)

    # define cohort donors and healthy sets
    donors <- meta_select %>% filter(Cohort == cohort & Group == cohort) %>% pull(KitID)
    healthies <- meta_select %>% filter(Cohort == cohort & Group == "Control") %>% pull(KitID)
    
    
    # iterate across donors
    for (d in donors){
        sub <- so %>% subset(KitID %in% c(d, healthies))
        sub$Group <- ifelse(sub$KitID == d, cohort, "Control")

        # ensure donor in object and sufficient cell numbers
        # calculate FC
        if (d %in% sub$KitID & table(sub$KitID)[d] > 30){
            degs <- FindMarkers(sub, ident.1 = cohort, ident.2 = "Control", logfc.threshold = 0, min.pct = 0.01, features = isgs_use,
                  group.by = "Group", assay = "RNA", test.use = "MAST") 
            degs$gene <- rownames(degs)
        
            degs %>% fwrite(file.path(wd, "Cohort_IFN_scores", "Cohort_DEGs", cohort, glue("{c}_{d}_MAST_degs.csv")))
            } 
        }
    }
}

### IFN Scoring

In [ ]:
source("IFN_Score_Generate.R")

#### L1/L2/PBMC

In [ ]:
celltype_level <- "L1"
if (celltype_level == "L1"){celltypes <- l1_celltypes}
if (celltype_level == "L2"){celltypes <- l2_celltypes}

In [109]:
# iterate across cohorts 
for (cohort in cohorts){
    dir.create(file.path(wd, "Cohort_IFN_scores", "NMF", cohort))
    print(cohort)
    donors <- meta %>% filter(Cohort == cohort & Group == cohort) %>% pull(KitID)
    print(donors)

    # iterate across celltypes
    for (c in celltypes) {
        print(c)
    
        
        # iterate donors and save in dataframe
        res_df <- map_dfr(donors, function(d){
                donor_df <- fread(file.path(wd, "Cohort_IFN_scores", "Cohort_DEGs", cohort, celltype_level, glue("{c}_{d}_MAST_degs.csv"))) %>% 
                        filter(pct.1 > 0.01 | pct.2 > 0.01)

                # nnls scoring function 
                ifn_score_generate(donor_df, 
                              cell_type_level = celltype_level,
                              cell_type = c,
                              stim_mat_dir = file.path(wd, "DEGs"))
                
            })

        # format results for export
        res_df_out <- t(res_df)
        colnames(res_df_out) <- paste(c, donors, sep = "_")
        res_df_out %>% fwrite(file.path(wd, "Cohort_IFN_scores", "NMF", cohort, glue("{cohort}_{c}_NMF_scores_scaled_v2.csv")))
    }
}

Warning message in dir.create(file.path(wd, "Cohort_IFN_scores", "NMF", cohort)):
“'/home/workspace/IFN//Cohort_IFN_scores/NMF/MM_BMMC_Internal_Healthy' already exists”


[1] "MM_BMMC_Internal_Healthy"
[1] "KT06477" "KT06476" "KT06475" "KT06478" "KT06480" "KT06479" "KT06853"
[8] "KT06482" "KT06481"
[1] "Tcell"


Warning message:
“Specifying the `id_cols` argument by position was deprecated in tidyr 1.3.0.
ℹ Please explicitly name `id_cols`, like `id_cols = !c(median_pval)`.”
x being coerced from class: matrix to data.table



[1] "Bcell"


Warning message:
“Specifying the `id_cols` argument by position was deprecated in tidyr 1.3.0.
ℹ Please explicitly name `id_cols`, like `id_cols = !c(median_pval)`.”
x being coerced from class: matrix to data.table



[1] "Monocyte"


Warning message:
“Specifying the `id_cols` argument by position was deprecated in tidyr 1.3.0.
ℹ Please explicitly name `id_cols`, like `id_cols = !c(median_pval)`.”
x being coerced from class: matrix to data.table



[1] "NK"


Warning message:
“Specifying the `id_cols` argument by position was deprecated in tidyr 1.3.0.
ℹ Please explicitly name `id_cols`, like `id_cols = !c(median_pval)`.”
x being coerced from class: matrix to data.table

